# EEG Person Identification using CNN + RNN - Complete Pipeline

## Overview
This notebook implements a complete deep learning pipeline for person identification using EEG signals:
1. **Data Preprocessing**: Load PhysioNet dataset, filter, segment, create spectrograms
2. **Model Training**: Build and train CNN+RNN hybrid architecture
3. **Performance Analysis**: Comprehensive evaluation and visualization

**Dataset**: PhysioNet Motor Movement/Imagery (109 subjects, 64-channel EEG)

**Architecture**: Conv2D → MaxPool → LSTM → Dense → Softmax (109 classes)

## Section 1: Setup and Installation

Install required libraries and import dependencies.

In [ ]:
# Install required packages (uncomment if running on Kaggle)
# !pip install mne mne-features scikit-learn scipy matplotlib seaborn h5py tqdm -q

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import pickle
import h5py
import json
from datetime import datetime
warnings.filterwarnings('ignore')

# EEG Processing
import mne
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf

# Signal Processing
from scipy import signal
from scipy.signal import stft
from scipy.stats import gaussian_kde

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import to_categorical

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, f1_score, precision_recall_fscore_support
)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU available: {len(gpus)} device(s)")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPU detected. Training will use CPU.")

print(f"\nTensorFlow version: {tf.__version__}")
print(f"MNE version: {mne.__version__}")
print("\nLibraries imported successfully!")

## Section 2: Configuration

Set all parameters for preprocessing, training, and evaluation.

In [ ]:
# Configuration parameters
CONFIG = {
    # Data parameters
    'n_subjects': 109,  # Total subjects in PhysioNet dataset
    'runs': [3, 7, 11],  # Motor imagery runs
    'n_subjects_to_use': 109,  # Reduce this for faster testing (e.g., 20)
    
    # Preprocessing parameters
    'sfreq': 160,  # Sampling frequency in Hz
    'l_freq': 0.5,  # Low-pass filter cutoff
    'h_freq': 50.0,  # High-pass filter cutoff
    'epoch_duration': 3.0,  # Duration of each epoch in seconds
    'overlap': 0.5,  # Overlap ratio between consecutive epochs
    
    # Time-frequency parameters
    'n_fft': 256,  # FFT window size for STFT
    'hop_length': 32,  # Hop length for STFT
    'n_freq_bins': 50,  # Number of frequency bins to keep
    
    # Data split parameters
    'train_ratio': 0.7,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    
    # Training parameters
    'batch_size': 32,
    'epochs': 100,  # Will stop early if validation loss plateaus
    'learning_rate': 0.001,
    
    # Output paths
    'data_dir': 'data',
    'processed_dir': 'data/processed',
    'model_dir': 'models',
    'figures_dir': 'figures',
    'logs_dir': 'logs'
}

# Create directories
for directory in [CONFIG['data_dir'], CONFIG['processed_dir'], CONFIG['model_dir'], 
                  CONFIG['figures_dir'], CONFIG['logs_dir']]:
    os.makedirs(directory, exist_ok=True)

print("Configuration set!")
print(f"\nPreprocessing Settings:")
print(f"  - Subjects: {CONFIG['n_subjects_to_use']}")
print(f"  - Sampling Rate: {CONFIG['sfreq']} Hz")
print(f"  - Bandpass Filter: {CONFIG['l_freq']}-{CONFIG['h_freq']} Hz")
print(f"  - Epoch Duration: {CONFIG['epoch_duration']} seconds")
print(f"\nTraining Settings:")
print(f"  - Batch Size: {CONFIG['batch_size']}")
print(f"  - Max Epochs: {CONFIG['epochs']}")
print(f"  - Learning Rate: {CONFIG['learning_rate']}")

## Section 3: Data Loading and Preprocessing

Load EEG data from PhysioNet and apply preprocessing.

In [ ]:
def load_subject_data(subject_id, runs, verbose=False):
    """
    Load EEG data for a single subject.
    """
    raw_fnames = eegbci.load_data(subject_id, runs, verbose=verbose)
    raw_list = [read_raw_edf(f, preload=True, verbose=verbose) for f in raw_fnames]
    raw = concatenate_raws(raw_list)
    eegbci.standardize(raw)
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing='ignore')
    return raw

def preprocess_raw(raw, l_freq=0.5, h_freq=50.0):
    """
    Apply bandpass filtering to raw EEG data.
    """
    raw_filtered = raw.copy()
    raw_filtered.filter(l_freq, h_freq, fir_design='firwin', verbose=False)
    return raw_filtered

def create_epochs(raw, epoch_duration=3.0, overlap=0.5):
    """
    Create fixed-duration epochs from continuous EEG data.
    """
    data = raw.get_data()
    sfreq = raw.info['sfreq']
    n_samples_per_epoch = int(epoch_duration * sfreq)
    step_size = int(n_samples_per_epoch * (1 - overlap))
    
    epochs_list = []
    n_channels, n_samples = data.shape
    
    for start_idx in range(0, n_samples - n_samples_per_epoch + 1, step_size):
        end_idx = start_idx + n_samples_per_epoch
        epoch = data[:, start_idx:end_idx]
        epochs_list.append(epoch)
    
    return np.array(epochs_list)

def compute_spectrograms(epochs_data, sfreq, n_fft=256, hop_length=32, 
                        n_freq_bins=50, expected_time_bins=None):
    """
    Compute spectrograms for all epochs and channels using STFT.
    """
    n_epochs, n_channels, n_times = epochs_data.shape
    spectrograms_list = []
    
    if expected_time_bins is None:
        f, t, Zxx = stft(epochs_data[0, 0], fs=sfreq, nperseg=n_fft, 
                        noverlap=n_fft-hop_length)
        expected_time_bins = Zxx.shape[1]
    
    for epoch in tqdm(epochs_data, desc="Computing spectrograms", leave=False):
        epoch_spectrograms = []
        
        for channel_data in epoch:
            f, t, Zxx = stft(channel_data, fs=sfreq, nperseg=n_fft, 
                            noverlap=n_fft-hop_length)
            magnitude = np.abs(Zxx)
            log_magnitude = np.log1p(magnitude)
            log_magnitude = log_magnitude[:n_freq_bins, :]
            
            # Ensure consistent time dimension
            current_time_bins = log_magnitude.shape[1]
            if current_time_bins < expected_time_bins:
                pad_width = expected_time_bins - current_time_bins
                log_magnitude = np.pad(log_magnitude, ((0, 0), (0, pad_width)), mode='edge')
            elif current_time_bins > expected_time_bins:
                log_magnitude = log_magnitude[:, :expected_time_bins]
            
            epoch_spectrograms.append(log_magnitude)
        
        spectrograms_list.append(np.array(epoch_spectrograms))
    
    return np.array(spectrograms_list)

print("Preprocessing functions defined!")

In [ ]:
# Test with one subject first
print("Testing data loading with Subject 1...")
test_raw = load_subject_data(1, CONFIG['runs'], verbose=False)
print(f"Loaded successfully!")
print(f"  - Channels: {len(test_raw.ch_names)}")
print(f"  - Duration: {test_raw.times[-1]:.2f} seconds")

# Preprocess
test_filtered = preprocess_raw(test_raw, CONFIG['l_freq'], CONFIG['h_freq'])

# Create epochs
test_epochs = create_epochs(test_filtered, CONFIG['epoch_duration'], CONFIG['overlap'])
print(f"\nEpochs shape: {test_epochs.shape}")

# Compute spectrograms
test_spectrograms = compute_spectrograms(
    test_epochs, 
    test_filtered.info['sfreq'],
    CONFIG['n_fft'],
    CONFIG['hop_length'],
    CONFIG['n_freq_bins']
)
print(f"Spectrograms shape: {test_spectrograms.shape}")

# Store expected time bins
CONFIG['expected_time_bins'] = test_spectrograms.shape[3]
print(f"Expected time bins: {CONFIG['expected_time_bins']}")

## Section 4: Visualize Sample Data

In [ ]:
# Visualize sample spectrograms
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

channels_to_plot = [0, 10, 20, 30, 40, 50]
epoch_idx = 5

for idx, ch_idx in enumerate(channels_to_plot):
    im = axes[idx].imshow(test_spectrograms[epoch_idx, ch_idx], 
                          aspect='auto', origin='lower', cmap='viridis')
    axes[idx].set_title(f'Channel {ch_idx}')
    axes[idx].set_xlabel('Time bins')
    axes[idx].set_ylabel('Frequency bins')
    plt.colorbar(im, ax=axes[idx])

plt.suptitle('Sample Spectrograms (Subject 1, Epoch 5)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'sample_spectrograms.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 5: Process All Subjects

**Note**: This will take 30-60 minutes depending on hardware and number of subjects.

In [ ]:
def process_all_subjects(n_subjects, config):
    """
    Process all subjects and create spectrograms.
    """
    all_spectrograms = []
    all_labels = []
    expected_time_bins = config.get('expected_time_bins', None)
    
    print(f"Processing {n_subjects} subjects...")
    if expected_time_bins is not None:
        print(f"Using fixed time bins: {expected_time_bins}\n")
    
    for subject_id in tqdm(range(1, n_subjects + 1), desc="Subjects"):
        try:
            raw = load_subject_data(subject_id, config['runs'], verbose=False)
            raw_filtered = preprocess_raw(raw, config['l_freq'], config['h_freq'])
            epochs = create_epochs(raw_filtered, config['epoch_duration'], config['overlap'])
            spectrograms = compute_spectrograms(
                epochs, raw_filtered.info['sfreq'],
                config['n_fft'], config['hop_length'], config['n_freq_bins'],
                expected_time_bins=expected_time_bins
            )
            
            # Normalize per subject
            mean = spectrograms.mean()
            std = spectrograms.std()
            spectrograms = (spectrograms - mean) / (std + 1e-8)
            
            all_spectrograms.append(spectrograms)
            all_labels.extend([subject_id - 1] * len(spectrograms))
            
        except Exception as e:
            print(f"\nError processing subject {subject_id}: {e}")
            continue
    
    X = np.concatenate(all_spectrograms, axis=0)
    y = np.array(all_labels)
    
    print(f"\nProcessing complete!")
    print(f"Total samples: {len(X)}")
    print(f"Shape: {X.shape}")
    
    return X, y

# Process all subjects
print("Starting full dataset processing...\n")
X_all, y_all = process_all_subjects(CONFIG['n_subjects_to_use'], CONFIG)

print(f"\nFinal dataset:")
print(f"  X shape: {X_all.shape}")
print(f"  y shape: {y_all.shape}")
print(f"  Unique subjects: {len(np.unique(y_all))}")

## Section 6: Train/Validation/Test Split

In [ ]:
def stratified_split(X, y, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=42):
    """
    Split data with stratification to ensure all subjects in all sets.
    """
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, train_size=train_ratio, stratify=y, random_state=random_state
    )
    
    val_size = val_ratio / (val_ratio + test_ratio)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, train_size=val_size, stratify=y_temp, random_state=random_state
    )
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Perform split
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
    X_all, y_all,
    CONFIG['train_ratio'],
    CONFIG['val_ratio'],
    CONFIG['test_ratio']
)

print("Data split complete!\n")
print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# Convert labels to one-hot encoding
n_subjects = len(np.unique(y_all))
y_train_cat = to_categorical(y_train, num_classes=n_subjects)
y_val_cat = to_categorical(y_val, num_classes=n_subjects)
y_test_cat = to_categorical(y_test, num_classes=n_subjects)

# Add channel dimension for CNN
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train_cat.shape}")

## Section 7: Build CNN + RNN Model

In [ ]:
def build_cnn_rnn_model(n_channels, n_freq_bins, n_time_bins, n_subjects):
    """
    Build CNN+RNN hybrid model for EEG person identification.
    """
    model = models.Sequential([
        # Input shape: (n_channels, n_freq_bins, n_time_bins, 1)
        layers.Input(shape=(n_channels, n_freq_bins, n_time_bins, 1)),
        
        # Reshape to merge channels with frequency for 2D CNN
        layers.Reshape((n_channels * n_freq_bins, n_time_bins, 1)),
        
        # CNN blocks
        layers.Conv2D(64, (5, 5), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Reshape for RNN
        layers.Reshape((-1, 128)),
        
        # RNN block
        layers.Bidirectional(layers.LSTM(128, return_sequences=False)),
        layers.Dropout(0.4),
        
        # Classification
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(n_subjects, activation='softmax')
    ], name='CNN_RNN_EEG_Identifier')
    
    return model

# Build model
print("Building CNN + RNN model...\n")
n_channels = X_train.shape[1]
n_freq_bins = X_train.shape[2]
n_time_bins = X_train.shape[3]

model = build_cnn_rnn_model(n_channels, n_freq_bins, n_time_bins, n_subjects)
model.summary()

# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
)

print("\nModel compiled successfully!")

## Section 8: Train Model

**Note**: Training may take 1-3 hours depending on hardware.

In [ ]:
# Setup callbacks
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = f"cnn_rnn_eeg_identifier_{timestamp}"
model_path = os.path.join(CONFIG['model_dir'], f"{model_name}.keras")
log_dir = os.path.join(CONFIG['logs_dir'], model_name)

callback_list = [
    callbacks.ModelCheckpoint(
        filepath=model_path,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1),
    callbacks.CSVLogger(os.path.join(CONFIG['model_dir'], f'{model_name}_training_log.csv'))
]

print(f"Starting training...")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Max epochs: {CONFIG['epochs']}")
print(f"  Model will be saved to: {model_path}\n")

# Train model
history = model.fit(
    X_train, y_train_cat,
    batch_size=CONFIG['batch_size'],
    epochs=CONFIG['epochs'],
    validation_data=(X_val, y_val_cat),
    callbacks=callback_list,
    verbose=1
)

print("\nTraining complete!")

## Section 9: Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy', fontsize=12)
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Top-5 Accuracy
axes[1, 0].plot(history.history['top5_accuracy'], label='Training Top-5 Acc', linewidth=2)
axes[1, 0].plot(history.history['val_top5_accuracy'], label='Validation Top-5 Acc', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Top-5 Accuracy', fontsize=12)
axes[1, 0].set_title('Top-5 Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate (if available)
if 'lr' in history.history:
    axes[1, 1].plot(history.history['lr'], linewidth=2, color='orange')
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print best metrics
best_epoch = np.argmax(history.history['val_accuracy'])
print(f"\nBest Validation Results (Epoch {best_epoch + 1}):")
print(f"  Loss: {history.history['val_loss'][best_epoch]:.4f}")
print(f"  Accuracy: {history.history['val_accuracy'][best_epoch]:.4f}")
print(f"  Top-5 Accuracy: {history.history['val_top5_accuracy'][best_epoch]:.4f}")

## Section 10: Model Evaluation

In [ ]:
# Load best model
print("Loading best model for evaluation...")
best_model = keras.models.load_model(model_path)

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = best_model.evaluate(X_test, y_test_cat, batch_size=CONFIG['batch_size'], verbose=1)

print(f"\nTest Set Results:")
print(f"  Loss: {test_results[0]:.4f}")
print(f"  Accuracy: {test_results[1]:.4f} ({test_results[1]*100:.2f}%)")
print(f"  Top-5 Accuracy: {test_results[2]:.4f} ({test_results[2]*100:.2f}%)")

# Get predictions
print("\nGenerating predictions...")
y_pred_probs = best_model.predict(X_test, batch_size=CONFIG['batch_size'], verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"\nDetailed Metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  F1-Score (Macro): {f1_macro:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted:.4f}")

## Section 11: Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Absolute counts
im1 = axes[0].imshow(cm, cmap='Blues', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Predicted Subject ID', fontsize=14)
axes[0].set_ylabel('True Subject ID', fontsize=14)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=16, fontweight='bold')
plt.colorbar(im1, ax=axes[0])

# Normalized
im2 = axes[1].imshow(cm_normalized, cmap='RdYlGn', aspect='auto', 
                     interpolation='nearest', vmin=0, vmax=1)
axes[1].set_xlabel('Predicted Subject ID', fontsize=14)
axes[1].set_ylabel('True Subject ID', fontsize=14)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=16, fontweight='bold')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 12: Per-Subject Performance Analysis

In [ ]:
# Per-subject metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)

# Create DataFrame
subject_metrics = []
for subject_id in range(n_subjects):
    subject_metrics.append({
        'Subject_ID': subject_id,
        'Test_Samples': support[subject_id],
        'Precision': precision[subject_id],
        'Recall': recall[subject_id],
        'F1_Score': f1[subject_id],
        'Accuracy': cm[subject_id, subject_id] / support[subject_id] if support[subject_id] > 0 else 0
    })

df_metrics = pd.DataFrame(subject_metrics)

print("Per-Subject Performance Summary:")
print(df_metrics.describe())

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# F1-Score by subject
axes[0, 0].bar(df_metrics['Subject_ID'], df_metrics['F1_Score'], alpha=0.7, edgecolor='black')
axes[0, 0].axhline(df_metrics['F1_Score'].mean(), color='red', linestyle='--', 
                   label=f'Mean: {df_metrics["F1_Score"].mean():.3f}')
axes[0, 0].set_xlabel('Subject ID', fontsize=12)
axes[0, 0].set_ylabel('F1-Score', fontsize=12)
axes[0, 0].set_title('F1-Score per Subject', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Precision vs Recall
scatter = axes[0, 1].scatter(df_metrics['Recall'], df_metrics['Precision'], 
                             alpha=0.6, s=100, c=df_metrics['F1_Score'], 
                             cmap='viridis', edgecolors='black')
axes[0, 1].plot([0, 1], [0, 1], 'r--', alpha=0.3)
axes[0, 1].set_xlabel('Recall', fontsize=12)
axes[0, 1].set_ylabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision vs Recall', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=axes[0, 1], label='F1-Score')
axes[0, 1].grid(alpha=0.3)

# Accuracy vs Sample Size
axes[1, 0].scatter(df_metrics['Test_Samples'], df_metrics['Accuracy'], 
                   alpha=0.6, s=100, c='coral', edgecolors='black')
axes[1, 0].set_xlabel('Number of Test Samples', fontsize=12)
axes[1, 0].set_ylabel('Accuracy', fontsize=12)
axes[1, 0].set_title('Accuracy vs Sample Size', fontsize=14, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Metrics distribution
df_metrics[['Precision', 'Recall', 'F1_Score']].boxplot(ax=axes[1, 1], patch_artist=True)
axes[1, 1].set_ylabel('Score', fontsize=12)
axes[1, 1].set_title('Distribution of Performance Metrics', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'per_subject_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save metrics
df_metrics.to_csv(os.path.join(CONFIG['figures_dir'], 'per_subject_metrics.csv'), index=False)
print(f"\nMetrics saved!")

## Section 13: t-SNE Visualization (Optional)

Visualize learned features in 2D space. This may take several minutes.

In [ ]:
# Extract features from the model
print("Extracting features for t-SNE visualization...")
feature_extractor = keras.Model(
    inputs=best_model.input,
    outputs=best_model.layers[-3].output
)

# Sample data for visualization (5 samples per subject)
n_samples_per_subject = 5
sample_indices = []
for subject_id in range(n_subjects):
    subject_mask = (y_test == subject_id)
    subject_indices = np.where(subject_mask)[0]
    if len(subject_indices) >= n_samples_per_subject:
        sample_indices.extend(np.random.choice(subject_indices, n_samples_per_subject, replace=False))
    else:
        sample_indices.extend(subject_indices)

X_sample = X_test[sample_indices]
y_sample = y_test[sample_indices]

print(f"Extracting features for {len(X_sample)} samples...")
features = feature_extractor.predict(X_sample, batch_size=32, verbose=0)

# Apply PCA then t-SNE
print("Applying PCA...")
pca = PCA(n_components=50)
features_pca = pca.fit_transform(features)

print("Applying t-SNE (this may take a few minutes)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=0)
features_tsne = tsne.fit_transform(features_pca)

# Visualize
plt.figure(figsize=(12, 10))
unique_subjects = np.unique(y_sample)
colors = plt.cm.tab20(np.linspace(0, 1, min(20, len(unique_subjects))))

for idx, subject_id in enumerate(unique_subjects[:20]):
    mask = (y_sample == subject_id)
    plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1], 
               c=[colors[idx]], label=f'Subject {subject_id}', 
               alpha=0.6, s=50, edgecolors='black', linewidth=0.5)

for subject_id in unique_subjects[20:]:
    mask = (y_sample == subject_id)
    plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1], 
               alpha=0.4, s=50, edgecolors='black', linewidth=0.5)

plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.title('t-SNE Visualization of Learned Features', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'tsne_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()

print("t-SNE visualization complete!")

## Section 14: Final Summary and Results

In [ ]:
# Generate comprehensive summary
print("="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"\nModel Architecture: CNN + RNN (Bidirectional LSTM)")
print(f"Total Parameters: {best_model.count_params():,}")
print(f"Number of Subjects: {n_subjects}")
print(f"\nDataset:")
print(f"  Training samples: {len(X_train)}")
print(f"  Validation samples: {len(X_val)}")
print(f"  Test samples: {len(X_test)}")
print(f"\nTest Performance:")
print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Top-5 Accuracy: {test_results[2]:.4f} ({test_results[2]*100:.2f}%)")
print(f"  F1-Score (Macro): {f1_macro:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted:.4f}")
print(f"\nPer-Subject Statistics:")
print(f"  Mean F1-Score: {df_metrics['F1_Score'].mean():.4f} ± {df_metrics['F1_Score'].std():.4f}")
print(f"  Best Subject F1: {df_metrics['F1_Score'].max():.4f}")
print(f"  Worst Subject F1: {df_metrics['F1_Score'].min():.4f}")
print(f"\nModel saved to: {model_path}")
print(f"Figures saved to: {CONFIG['figures_dir']}/")
print("="*80)

# Save final results
final_results = {
    'model_name': model_name,
    'architecture': 'CNN + RNN (LSTM)',
    'n_subjects': n_subjects,
    'n_parameters': int(best_model.count_params()),
    'test_metrics': {
        'accuracy': float(accuracy),
        'top5_accuracy': float(test_results[2]),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted)
    },
    'per_subject_stats': {
        'mean_f1': float(df_metrics['F1_Score'].mean()),
        'std_f1': float(df_metrics['F1_Score'].std()),
        'min_f1': float(df_metrics['F1_Score'].min()),
        'max_f1': float(df_metrics['F1_Score'].max())
    }
}

results_file = os.path.join(CONFIG['model_dir'], f'{model_name}_results.json')
with open(results_file, 'w') as f:
    json.dump(final_results, f, indent=4)

print(f"\nResults saved to: {results_file}")
print("\n✓ Complete pipeline finished successfully!")

## Notes and Tips for Kaggle

### Running on Kaggle:
1. **Enable GPU**: Go to Settings → Accelerator → GPU for faster training
2. **Reduce subjects for testing**: Set `n_subjects_to_use = 20` in CONFIG for quick testing
3. **Save outputs**: Download the `figures/` and `models/` folders after completion

### Customization:
- Adjust `batch_size` if you encounter memory errors (try 16 or 64)
- Modify `epochs` to change training duration
- Change `n_subjects_to_use` to work with fewer subjects for faster experimentation
- Experiment with model architecture in the `build_cnn_rnn_model` function

### Expected Runtime:
- **Preprocessing (109 subjects)**: 30-60 minutes
- **Training (GPU)**: 1-2 hours
- **Training (CPU)**: 4-6 hours
- **Evaluation**: 5-10 minutes

### Troubleshooting:
- **Memory errors**: Reduce `n_subjects_to_use` or `batch_size`
- **Slow training**: Enable GPU in Kaggle settings
- **Import errors**: Uncomment and run the pip install cell at the top